# Reviewer analyses for data-4668825
Run the cells from top to bottom with **Shift+Enter**. Wait until `[*]` on the left becomes a number before running the next cell.

## Step 1. Go to the project folder and check the scripts

In [ ]:
import os
PROJECT_ROOT = r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
os.chdir(PROJECT_ROOT)
print("Working folder:", os.getcwd())
for f in ["reviewer_R1_analyses.py", "reviewer_R2_analyses.py"]:
    print(f, "found" if os.path.exists(os.path.join("10_revision", f)) else "NOT FOUND")

## Step 2. Install missing packages (once)
After this cell finishes, use **Kernel > Restart** and run Step 1 again.

In [ ]:
%pip install statsmodels xgboost pyarrow

## Step 3. Quick check (a few minutes, no model refitting)
Expected: S3 true_positives = 75, 229, 363, 562, 731; Q7 roc T4-T9 = 0.7508, 0.7597, 0.7294, 0.7235, 0.7168, 0.7498 and T10 = 0.7432.

In [ ]:
%run 10_revision/reviewer_R2_analyses.py --only Q7 S3

In [ ]:
import os, re, json, pandas as pd, pyarrow.parquet as pq

# 1) Columns of the feature table
cols = pq.ParquetFile("02_intermediate/modelling/modelling_panel_T1_T10.parquet").schema_arrow.names
print(len(cols), "columns in the feature table:\n")
print(cols)

# 2) Files that may hold the frozen A5 predictor list
print("\n\nCandidate specification files (number of feature-table columns named in each):")
hits = []
for root in ["05_results", "04_models", "06_frozen", "02_intermediate", "07_provenance", "10_revision_out"]:
    if not os.path.exists(root):
        continue
    for dp, dn, fn in os.walk(root):
        for f in fn:
            if f.lower().endswith((".json", ".yaml", ".yml", ".txt", ".csv")) and os.path.getsize(os.path.join(dp, f)) < 5e6:
                p = os.path.join(dp, f)
                try:
                    txt = open(p, encoding="utf-8", errors="ignore").read()
                except Exception:
                    continue
                n = sum(1 for c in cols if re.search(r"(?<![\w])" + re.escape(c) + r"(?![\w])", txt))
                if n >= 40:
                    hits.append((n, p))
for n, p in sorted(hits, reverse=True)[:15]:
    print(n, p)

In [ ]:
print(open("10_revision_out/RV_reviewer2/RV2_SUMMARY.txt", encoding="utf-8").read())

**Stop here and send the output above to Claude before continuing.**

## Step 4. Reviewer 1, fast parts (RV3, RV7)

In [ ]:
%run 10_revision/reviewer_R1_analyses.py --only RV3 RV7

In [ ]:
print(open("10_revision_out/RV_reviewer1/RV_SUMMARY.txt", encoding="utf-8").read())

In [ ]:
import os, re
for root in ["05_results", "04_models", "02_intermediate"]:
    if not os.path.exists(root):
        continue
    for dp, dn, fn in os.walk(root):
        for f in fn:
            p = os.path.join(dp, f)
            if re.search(r"logit|logistic|LR_|_lr", f, re.I):
                print("FILE NAME:", p)
            elif f.lower().endswith((".csv", ".json", ".txt", ".md")) and os.path.getsize(p) < 5e6:
                try:
                    txt = open(p, encoding="utf-8", errors="ignore").read()
                except Exception:
                    continue
                if re.search(r"logistic|LogisticRegression|\blogit\b", txt, re.I):
                    print("MENTIONS:", p)

In [ ]:
import pandas as pd, json
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50); pd.set_option("display.max_colwidth", 80)
D = "05_results/development_modelling/"

print("=== SUMMARY ===")
print(pd.read_csv(D + "primary_development_metrics_summary.csv").to_string())

print("\n=== BY WINDOW ===")
print(pd.read_csv(D + "primary_development_metrics_by_window.csv").to_string())

print("\n=== MANIFEST (lines mentioning logistic / M1 / features) ===")
txt = open(D + "development_modelling_manifest.json", encoding="utf-8").read()
for line in txt.splitlines():
    if any(k in line.lower() for k in ["logistic", "m1", "n_features", "feature_set", "predictor", "structured", "window", "train"]):
        print(line[:200])

In [ ]:
%run 10_revision/reviewer_R2_analyses.py --only Q5 Q9 Q10

In [ ]:
import os
os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
print("Working folder:", os.getcwd())
print("Script found:", os.path.exists("10_revision/reviewer_R2_analyses.py"))
%run 10_revision/reviewer_R2_analyses.py --only Q1 Q3

## Step 5. Reviewer 1, all analyses (long: refits the model many times)
For Comment 5, first set `dwellings_file` in `reviewer_R1_analyses.py` to the Census CSV in `01_raw/census/`.

In [ ]:
%run 10_revision/reviewer_R1_analyses.py

## Step 6. Reviewer 2, all analyses except tuning

In [ ]:
%run 10_revision/reviewer_R2_analyses.py

## Step 7. Reviewer 2, hyperparameter grid (longest, 2-5 hours)

In [ ]:
%run 10_revision/reviewer_R2_analyses.py --only Q6 --tune

## Step 8. Show both summaries (send these two files to Claude)

In [ ]:
for p in ["10_revision_out/RV_reviewer1/RV_SUMMARY.txt", "10_revision_out/RV_reviewer2/RV2_SUMMARY.txt"]:
    print("=" * 30, p)
    print(open(p, encoding="utf-8").read() if os.path.exists(p) else "not created yet")

In [ ]:
# ===== OVERNIGHT RUN: all remaining reviewer analyses =====
import os, shutil, time, traceback
os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
print("Working folder:", os.getcwd())
print("R1 script found:", os.path.exists("10_revision/reviewer_R1_analyses.py"))
print("R2 script found:", os.path.exists("10_revision/reviewer_R2_analyses.py"))

SAVE = "10_revision_out/overnight"
os.makedirs(SAVE, exist_ok=True)
ip = get_ipython()

jobs = [
    ("R1_main", "10_revision/reviewer_R1_analyses.py --only RV1 RV2 RV4 RV5 RV6",
     "10_revision_out/RV_reviewer1/RV_SUMMARY.txt"),
    ("R2_Q4_Q8", "10_revision/reviewer_R2_analyses.py --only Q4 Q8",
     "10_revision_out/RV_reviewer2/RV2_SUMMARY.txt"),
    ("R2_Q6_tuning", "10_revision/reviewer_R2_analyses.py --only Q6 --tune",
     "10_revision_out/RV_reviewer2/RV2_SUMMARY.txt"),
]

with open(f"{SAVE}/STATUS.txt", "w", encoding="utf-8") as status:
    for name, cmd, summary in jobs:
        t0 = time.time()
        msg = f"{time.strftime('%Y-%m-%d %H:%M')}  START {name}"
        print("\n" + "#" * 90 + "\n" + msg); status.write(msg + "\n"); status.flush()
        try:
            ip.run_line_magic("run", cmd)
            result = "OK"
        except BaseException as e:
            result = f"FAILED: {type(e).__name__}: {e}"
            with open(f"{SAVE}/{name}_ERROR.txt", "w", encoding="utf-8") as f:
                f.write(traceback.format_exc())
        if os.path.exists(summary):
            shutil.copy(summary, f"{SAVE}/{name}_SUMMARY.txt")
        msg = f"{time.strftime('%Y-%m-%d %H:%M')}  END {name}: {result} ({(time.time() - t0) / 60:.0f} min)"
        print(msg); status.write(msg + "\n"); status.flush()

print("\nALL DONE. Results are in", os.path.abspath(SAVE))

In [ ]:
import os, shutil
os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
print("Working folder:", os.getcwd())
print("Census files:", os.listdir("01_raw/census") if os.path.exists("01_raw/census") else "FOLDER NOT FOUND")
print("Q6 summary already saved:", os.path.exists("10_revision_out/overnight/R2_Q6_tuning_SUMMARY.txt"))

# Q8 (fixed script)
%run 10_revision/reviewer_R2_analyses.py --only Q8
shutil.copy("10_revision_out/RV_reviewer2/RV2_SUMMARY.txt", "10_revision_out/overnight/R2_Q8_SUMMARY.txt")

# RV5 (uses the Census file)
%run 10_revision/reviewer_R1_analyses.py --only RV5
shutil.copy("10_revision_out/RV_reviewer1/RV_SUMMARY.txt", "10_revision_out/overnight/R1_RV5_SUMMARY.txt")
print("ALL DONE")

In [ ]:
import os; os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
%run 11_final/copy_revision_outputs.py

In [ ]:
import os; os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
%run 11_final/build_supplementary.py

In [ ]:
import os; os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
%run 11_final/copy_revision_outputs.py

In [ ]:
import os, shutil; os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
shutil.copy("10_revision_out/RV_reviewer1/RV_SUMMARY.txt", "10_revision_out/RV_reviewer1/RV_SUMMARY_backup.txt")
%run 10_revision/reviewer_R1_analyses.py --only RV3 RV6 RV7

In [ ]:
import os; os.chdir(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
%run 11_final/copy_revision_outputs.py

In [ ]:
%run 11_final/build_supplementary.py